# 01 — Veri denetimi (Faz 0)

**Bilimsel amaç:** Model eğitiminden önce (1) klinik tabloyu, (2) ses klasörünü, (3) katılımcı ↔ kayıt ↔ görev eşlemesini, (4) kopyaları ve **kayıt koşullarının zamanla değişip değişmediğini** sayılarla doğrulamak.

**Bu notebook model EĞİTMEZ.** GPU gerekmez: *Runtime → Change runtime type → CPU* seçerek GPU kotanı koru.

**Ön koşullar** (bkz. `docs/COLAB_WORKFLOW.md`):
- Drive'da `MyDrive/asthma-voice/data_raw/` içinde `soundData.zip`, `clinical_data.csv`, `astim-tarama_Data Report_20260314.xlsx`
- Colab Secrets'ta `GITHUB_TOKEN`

**Dosya adlandırması (veri ekibi):** `<katılımcı ID>_<slot>.m4a` — ör. `101001_2.m4a` = 2. kayıt = araba. 48 kHz mono, iPhone 14, 10 cm, hep aynı yer.

**Çıktılar:**
- Drive `data_derived/`: `participants.csv`, `audio_inventory.csv`, `recording_map.csv`, `near_duplicate_pairs.csv` (katılımcı düzeyi → git'e girmez)
- Repo `reports/`: `clinical_audit.*`, `audio_audit.*` (agrega → git'e girebilir) ve `configs/slot_task_map.yaml`

## 0. Oturum kurulumu
### 0.1 Ortam kontrolü

In [ ]:
import sys, platform, subprocess
print("Python:", sys.version.split()[0], "|", platform.platform())
print(subprocess.run(["ffmpeg", "-version"], capture_output=True, text=True).stdout.splitlines()[0])
import numpy, pandas, sklearn, scipy
print("numpy", numpy.__version__, "| pandas", pandas.__version__, "| sklearn", sklearn.__version__, "| scipy", scipy.__version__)

### 0.2 Drive'ı bağla

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

### 0.3 Yollar — yalnızca bu hücrede değişiklik yap

In [ ]:
from pathlib import Path
DRIVE_ROOT = Path('/content/drive/MyDrive/asthma-voice')
RAW        = DRIVE_ROOT / 'data_raw'
DERIVED    = DRIVE_ROOT / 'data_derived'
GITHUB_USER = 'BURAYA_GITHUB_KULLANICI_ADIN'   # <-- değiştir
REPO_NAME   = 'asthma-voice-dl'
REPO_DIR    = Path('/content') / REPO_NAME
AUDIO_ZIP   = RAW / 'soundData.zip'
AUDIO_DIR   = Path('/content/audio')            # zip buraya, YEREL diske açılır
CSV  = RAW / 'clinical_data.csv'
XLSX = RAW / 'astim-tarama_Data Report_20260314.xlsx'

for p in [RAW, CSV, XLSX, AUDIO_ZIP]:
    assert p.exists(), f'Bulunamadı: {p}  -> Drive klasör yapısını kontrol et'
DERIVED.mkdir(parents=True, exist_ok=True)
assert GITHUB_USER != 'BURAYA_GITHUB_KULLANICI_ADIN', 'GITHUB_USER değişkenini doldur'
print('Tüm girdiler mevcut.')

### 0.4 Repo'yu klonla ya da güncelle (token Secrets'tan okunur, ekrana yazılmaz)

In [ ]:
import subprocess
from google.colab import userdata
token = userdata.get('GITHUB_TOKEN')
url = f'https://{token}@github.com/{GITHUB_USER}/{REPO_NAME}.git'
if not REPO_DIR.exists():
    r = subprocess.run(['git', 'clone', url, str(REPO_DIR)], capture_output=True, text=True)
else:
    r = subprocess.run(['git', '-C', str(REPO_DIR), 'pull'], capture_output=True, text=True)
print((r.stdout + r.stderr).replace(token, '***'))
assert r.returncode == 0, 'git başarısız — token / kullanıcı adı / repo adı?'
commit = subprocess.run(['git', '-C', str(REPO_DIR), 'rev-parse', 'HEAD'], capture_output=True, text=True).stdout.strip()
print('Repo commit:', commit)

### 0.5 Paketler

In [ ]:
!pip install -q -r {REPO_DIR}/requirements-colab.txt

### 0.6 Ses zip'ini yerel diske kopyala, hash'ini al, aç
Hash = veri sürümü. EXPERIMENTS.md'ye ve registry'ye yazılacak.

In [ ]:
import hashlib, shutil, zipfile, time
from collections import Counter
def sha256(path, chunk=1 << 20):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for b in iter(lambda: f.read(chunk), b''):
            h.update(b)
    return h.hexdigest()

t0 = time.time()
local_zip = Path('/content/soundData.zip')
if not local_zip.exists():
    shutil.copy(AUDIO_ZIP, local_zip)
AUDIO_ZIP_SHA256 = sha256(local_zip)
print('soundData.zip sha256:', AUDIO_ZIP_SHA256, f'({local_zip.stat().st_size/1e6:.1f} MB, {time.time()-t0:.0f} s)')
if not AUDIO_DIR.exists():
    with zipfile.ZipFile(local_zip) as z:
        z.extractall(AUDIO_DIR)
files = [p for p in AUDIO_DIR.rglob('*') if p.is_file()]
print('Toplam dosya:', len(files), '| uzantılar:', Counter(p.suffix.lower() for p in files))
print('Örnek yollar:', [str(p.relative_to(AUDIO_DIR)) for p in sorted(files)[:5]])
print('Beklenen: 344 katılımcı × 7 = 2408 ses dosyası')

## 1. Klinik denetim (EXP-001 dahil)

In [ ]:
!cd {REPO_DIR} && python scripts/audit_clinical.py --csv "{CSV}" --xlsx "{XLSX}" --out-private "{DERIVED}" --out-report reports/clinical_audit

## 2. Ses denetimi + eşleme
Dosyaların çoğu `<ID>_<slot>` kalıbına uymazsa script **çıkış kodu 2** ile durur ve dosya listesini gösterir. Bu bir hata değil, bilinçli bir güvenlik önlemi: eşlemeyi tahminle yapmayız.
Süre: 2408 dosya için CPU'da yaklaşık 5–15 dk.

In [ ]:
r = subprocess.run(['python', 'scripts/audit_audio.py', '--audio-root', str(AUDIO_DIR),
                    '--participants', str(DERIVED / 'participants.csv'),
                    '--out-private', str(DERIVED), '--out-report', 'reports/audio_audit'],
                   cwd=REPO_DIR, capture_output=True, text=True)
print(r.stdout[-8000:]); print(r.stderr[-3000:])
print('ÇIKIŞ KODU:', r.returncode, '(0 = tamam, 2 = adlandırma tanınmadı, eşleme yapılmadı)')

### 2.1 En önemli üç kontrol
1. **Eşleme:** makale kohortunun (344) hepsinde 7 kaydın tamamı OK mi? Dosyası hiç olmayanlar kimler (beklenti: 101345–101348)?
2. **Meta veri tekdüze mi?** Uzantı, codec, bitrate, cihaz modeli, iOS sürümü… Bunlardan biri **hastalarda kayıt dönemine göre** değişiyorsa, zamansal confounder'ın akustik yolu somutlaşır.
3. **Dosyadaki kayıt zamanı CSV tarihiyle uyuşuyor mu?** Uyuşuyorsa dosya tarihi bağımsız bir doğrulamadır; CSV'de tarihi eksik 3 sağlıklının tarihini de verir.

In [ ]:
import json
A = json.loads((REPO_DIR / 'reports/audio_audit/audio_audit.json').read_text())
mp = A['mapping']
print('Makale kohortunda 7/7 OK:', mp['paper_cohort_all_7_ok'], '/', mp['paper_cohort_size'])
print('Hiç dosyası olmayan:', mp['participants_with_zero_files'])
print('Eksik/çoklu kayıtlı katılımcılar:', mp['participants_partial'])
print('Durumlar:', mp['status_counts'])
print()
for col, e in A['metadata_vs_label_and_period'].items():
    print(f"{col:22s}", 'tekdüze ✔ ' + str(e['value']) if e['uniform'] else
          f"DEĞİŞKEN ⚠ etiket p={e.get('label_chi2_p')}, hastalarda dönem p={e.get('period_chi2_p')}")
print()
print('Dosya zamanı vs CSV tarihi:', A.get('creation_time_vs_csv_date'))
print('Gizlilik:', A.get('privacy_warning', 'konum etiketi yok'))

## 3. Slot → görev eşlemesini dinleyerek TEYİT ET
Veri ekibine göre (düzeltilmiş): **1 = aaa, 2 = araba, 3 = ana, 4 = ordu, 5 = gelecek, 6 = titiz, 7 = ünlem**. Her slottan sabit tohumla 3 kayıt dinle (~5 dk) ve duyduğunla uyuşuyor mu, işaretle. Amaç tahmin değil, yanlış bir kopyala-yapıştırın görev bazlı tüm analizleri sessizce bozmasını önlemek.

In [ ]:
import pandas as pd, numpy as np, tempfile
from IPython.display import Audio, display, Markdown
M = pd.read_csv(DERIVED / 'recording_map.csv')
ok = M[M['status'] == 'OK']
SLOT_TASK = {1: 'aaa', 2: 'araba', 3: 'ana', 4: 'ordu', 5: 'gelecek', 6: 'titiz', 7: 'ünlem'}

def play(relpath):
    # Tarayıcının çalabileceği WAV'a çevir (orijinal dosyaya dokunmaz, konum dahil meta veriyi atar)
    out = Path(tempfile.gettempdir()) / (Path(relpath).stem + '.wav')
    subprocess.run(['ffmpeg', '-v', 'error', '-y', '-i', str(AUDIO_DIR / relpath), '-map', '0:a:0',
                    '-map_metadata', '-1', str(out)], check=True)
    display(Audio(str(out)))

for slot in range(1, 8):
    d = ok[ok['slot'] == slot]
    display(Markdown(f"### Slot {slot} — beklenen: **{SLOT_TASK[slot]}**  (OK kayıt: {len(d)})"))
    for _, row in d.sample(n=min(3, len(d)), random_state=slot).iterrows():
        print(f"{row['relpath']}  süre {row['duration_s']:.1f} s")
        play(row['relpath'])

In [ ]:
# Dinledikten sonra doldur: 21 kaydın hepsi beklenen görevle uyuşuyorsa True, en az biri uyuşmuyorsa False
LISTENING_CONSISTENT = None
NOTES = ''   # uyuşmazlık varsa hangi dosya, ne duydun
assert LISTENING_CONSISTENT in (True, False), 'Önce dinle, sonra LISTENING_CONSISTENT değerini True ya da False yap'
import yaml
cfg = {'status': 'confirmed_by_listening' if LISTENING_CONSISTENT else 'CONFLICT_FOUND',
       'source': 'veri ekibi, düzeltilmiş bildirim (2026-10-08) + dinleme teyidi',
       'listened_per_slot': 3, 'sample_seed': 'slot numarası', 'notes': NOTES, 'slot_to_task': SLOT_TASK}
(REPO_DIR / 'configs' / 'slot_task_map.yaml').write_text(yaml.safe_dump(cfg, allow_unicode=True, sort_keys=False))
print(open(REPO_DIR / 'configs' / 'slot_task_map.yaml').read())
assert LISTENING_CONSISTENT, 'Uyuşmazlık var: görev bazlı analizlere geçmeden veri ekibiyle netleştir'

## 4. Kimlik kontrolü: 101042 (astım) ve 101050 (sağlıklı) aynı kişi mi?
Aynı doğum tarihi + cinsiyet, ikisi de sigara içiyor, 4 gün arayla kaydedilmiş. Şans eseri böyle çiftler beklenir (rapor Bölüm 4) ama farklı etiketli tek çift bu; 1 dakikalık dinleme yeterli.

In [ ]:
for pid in [101042, 101050]:
    row = ok[(ok['participant_id'] == pid) & (ok['slot'] == 4)]
    if len(row):
        print(pid); play(row.iloc[0]['relpath'])

## 5. Kaydet
Yalnız agrega raporlar ve slot eşlemesi commit edilir. Önce **Edit → Clear all outputs** (çıktılarda dosya adları var).
```bash
cd /content/asthma-voice-dl && git status
git add reports/ configs/slot_task_map.yaml && git commit -m "audit: ses denetimi + slot eşlemesi teyit edildi" && git push
```
Sonra şunları paylaş: `reports/audio_audit/audio_audit.md` içeriği, `AUDIO_ZIP_SHA256`, dinleme teyidinin sonucu ve kimlik kontrolü sonucu.